# DSA5208 Assignment 1

### Group Members

-

## Setup

In [ ]:
# Only run if needed
# %pip install pymongo

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.errors import ExecutionTimeout, PyMongoError
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

import time
import pandas as pd

## Connections

- `client` connects to the replica set `dsa5208`. Writes always go to the primary.
- `client3` connects directly to `mongo3`. This will be used to pause mongo3's replication and to inspect mongo3's copy of the data.

Write and read concerns are applied in the `CONFIGS` below for scalability.

In [2]:
REPLICA_SET_URI = "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208"
MONGO3_URI = "mongodb://mongo3:27017/?directConnection=true"
DB_NAME = "university"

client = MongoClient(REPLICA_SET_URI, serverSelectionTimeoutMS=5000)
client3 = MongoClient(MONGO3_URI, serverSelectionTimeoutMS=5000)

## Consistency Configurations

Each entry is one combination we test. Adding a combination is a one-line change.

In [3]:
CONFIGS = {
    "majority": {"write_concern": "majority", "read_concern": "majority"},
    "majority-local": {"write_concern": "majority", "read_concern": "local"},
    "w1-majority": {"write_concern": 1, "read_concern": "majority"},
    "w1-local": {"write_concern": 1, "read_concern": "local"}
}

READ_TIMEOUT_MS = 3000   # how long a read may wait before we record TIMEOUT
NUM_READS = 10           # reads per loop, so we hit both secondaries
CATCHUP_SECONDS = 2      # safety margin for mongo3 to catch up after resuming
TIMEOUT = "TIMEOUT"

## Helper Functions

### Cluster status

In [27]:
def show_members():
    """Print each member's state and whether it has applied everything the primary has."""
    members = client.admin.command("replSetGetStatus")["members"]
    primary = next((m for m in members if m["stateStr"] == "PRIMARY"), None)

    print("\n")
    
    for m in members:
        print(m["name"], m["stateStr"], m.get("optimeDate"))

    print("\n")


def check_cluster():
    """Confirm the replica set is healthy and mongo3 is a secondary."""
    hello = client.admin.command("hello")
    print("Replica set:\t\t", hello.get("setName"))
    print("Primary:\t\t", hello.get("primary"))
    print("Writable primary:\t", hello.get("isWritablePrimary"))
    show_members()

    # The experiments pause mongo3, which only makes sense if it is a secondary.
    assert hello.get("primary") != "mongo3:27017", (
        "mongo3 is currently primary. Step it down before running the experiments."
    )

### Collections for a given configuration

In [5]:
def get_collection(collection_name, config_name, target="secondary"):
    """
    Return a collection handle with the config's write and read concern applied.

    target:
        "primary"   - reads come from the primary
        "secondary" - reads come from any secondary (mongo2 or mongo3)
        "mongo3"    - reads come directly from mongo3, using local read concern,
                      so we see exactly what mongo3 has stored

    Writes always go to the primary, whatever the target.
    """
    config = CONFIGS[config_name]

    if target == "mongo3":
        db = client3.get_database(DB_NAME, read_concern=ReadConcern("local"))
        return db[collection_name]

    read_preference = {
        "primary": ReadPreference.PRIMARY,
        "secondary": ReadPreference.SECONDARY,
    }[target]

    db = client.get_database(
        DB_NAME,
        write_concern=WriteConcern(w=config["write_concern"]),
        read_concern=ReadConcern(config["read_concern"]),
        read_preference=read_preference,
    )
    return db[collection_name]

### Stalling Mongo3 node

In [6]:
def stall_mongo3():
    """Pause replication on mongo3."""
    client3.admin.command({
        "configureFailPoint": "rsSyncApplyStop",
        "mode": "alwaysOn"
    })
    time.sleep(2)


def unstall_mongo3():
    """Resume replication on mongo3."""
    client3.admin.command({
        "configureFailPoint": "rsSyncApplyStop",
        "mode": "off"
    })

### Test data

Test documents are reset with `w=3`, which waits until **all three** nodes have the baseline. This guarantees mongo3 starts from the correct state before we pause it, without relying on manual delays.

Each config uses its own document IDs (e.g. `S001-majority`) so results from different configs never mix.

In [7]:
SETUP_WRITE_CONCERN = WriteConcern(w=3, wtimeout=10000)


def reset_document(collection_name, id_field, document):
    """Create or overwrite a test document and wait until every node has it."""
    db = client.get_database(DB_NAME, write_concern=SETUP_WRITE_CONCERN)
    db[collection_name].replace_one({id_field: document[id_field]}, document, upsert=True)


def make_student(student_id, name, **overrides):
    student = {
        "student_id": student_id,
        "student_name": name,
        "student_status": "Undergraduate",
        "seniority": "Year 1",
        "academic_load": "Full Time",
    }
    student.update(overrides)
    return student


def make_course(course_id, name):
    return {
        "course_id": course_id,
        "course_name": name,
        "course_status": "ACTIVE",
        "vacancy": 30,
    }

### Reading and reporting

In [26]:
def read_field(collection, query, field, session=None):
    """
    Read one field of one document.
    Returns TIMEOUT if the node could not serve the read within READ_TIMEOUT_MS
    (this is how a causal session blocks a stale read).
    """
    try:
        doc = collection.find_one(query, session=session, max_time_ms=READ_TIMEOUT_MS)
    except ExecutionTimeout:
        return TIMEOUT
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def read_field_repeatedly(collection, query, field, session=None, n=NUM_READS):
    return [read_field(collection, query, field, session) for _ in range(n)]


def show(label, value):
    print(f"{label:<30} {value}")


def show_reads(label, reads):
    """Print the reads as a table, one row per read."""
    table = pd.DataFrame({"read": range(1, len(reads) + 1), "value": reads})
    print(f"{label}")
    print(table.to_string(index=False))


def header(title, config_name):
    config = CONFIGS[config_name]
    print(f"=== {title} | w={config['write_concern']}, "
          f"readConcern={config['read_concern']} ===")


def report_stale_reads(reads, expected):
    """Read-your-writes check: any value other than what we wrote is stale."""
    stale = sum(1 for r in reads if r not in (expected, TIMEOUT))
    timeouts = reads.count(TIMEOUT)
    verdict = "VIOLATION" if stale else "no violation in this run"
    print(f"RESULTS: {verdict}: {stale} stale, {timeouts} timed out, out of {len(reads)} reads")


def report_regressions(reads, order):
    """Monotonic-reads check: count reads that went back to an older value."""
    rank = {value: i for i, value in enumerate(order)}
    newest_seen = -1
    regressions = 0
    for value in reads:
        if value not in rank:  # TIMEOUT or error
            continue
        if rank[value] < newest_seen:
            regressions += 1
        newest_seen = max(newest_seen, rank[value])

    timeouts = reads.count(TIMEOUT)
    verdict = "VIOLATION" if regressions else "no violation in this run"
    print(f"RESULTS: {verdict}: {regressions} went backwards, {timeouts} timed out, "
          f"out of {len(reads)} reads")

## Experiment Functions

Experiment Flow:

1. Reset all nodes.
2. Pause mongo3.
3. Run the operations **without** a session and record what we read.
4. Run the operations **with** a causally consistent session and record what we read.
5. Resume mongo3.

In [9]:
def run_read_your_writes(config_name):
    header("Read-your-writes", config_name)
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S001-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(student_id, "Robert Tan"))

    stall_mongo3()
    try:
        # Without a session
        students_secondary.update_one(query, {"$set": {"student_status": "Graduate"}})
        show("Wrote student_status:", "Graduate")
        reads = read_field_repeatedly(students_secondary, query, "student_status")
        show_reads("Reads (no session):", reads)
        report_stale_reads(reads, expected="Graduate")

        show_members()

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            students_secondary.update_one(
                query, {"$set": {"student_status": "Post Doc"}}, session=session
            )
            show("Wrote student_status:", "Post Doc")
            reads = read_field_repeatedly(
                students_secondary, query, "student_status", session=session
            )
            show_reads("Reads (causal consistency session):", reads)
            report_stale_reads(reads, expected="Post Doc")
    except:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

In [10]:
SENIORITY_ORDER = ["Year 1", "Year 2", "Year 3", "Year 4", "Year 5"]


def run_monotonic_reads(config_name):
    header("Monotonic reads", config_name)
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S002-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(student_id, "Feyd Johnston"))

    stall_mongo3()
    try:
        for year in SENIORITY_ORDER[1:]:
            students_secondary.update_one(query, {"$set": {"seniority": year}})
        show("Wrote seniority:", " -> ".join(SENIORITY_ORDER))

        # Without a session
        reads = read_field_repeatedly(students_secondary, query, "seniority")
        show_reads("Reads (no session):", reads)
        report_regressions(reads, SENIORITY_ORDER)

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            reads = read_field_repeatedly(
                students_secondary, query, "seniority", session=session
            )
        show_reads("Reads (causal consistency session):", reads)
        report_regressions(reads, SENIORITY_ORDER)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

In [11]:
def run_monotonic_writes(config_name):
    header("Monotonic writes", config_name)
    students_primary = get_collection("students", config_name, "primary")
    students_secondary = get_collection("students", config_name, "secondary")
    students_mongo3 = get_collection("students", config_name, "mongo3")

    student_id = f"S003-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(
        student_id, "Alan White",
        student_status="Post Doc",
        academic_history=["B.Sc. Computer Science"],
    ))

    stall_mongo3()
    try:
        for degree in ["M.Sc. Data Science", "PhD Information Systems"]:
            students_primary.update_one(query, {"$push": {"academic_history": degree}})

        show("Primary:", read_field(students_primary, query, "academic_history"))
        show("mongo3 (stalled):", read_field(students_mongo3, query, "academic_history"))

        reads = read_field_repeatedly(students_secondary, query, "academic_history")
        show_reads("Reads from secondaries (no session):", reads)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

    time.sleep(CATCHUP_SECONDS)
    show("mongo3 (resumed):", read_field(students_mongo3, query, "academic_history"))

In [12]:
def run_writes_follow_reads(config_name):
    header("Writes-follow-reads", config_name)
    courses_secondary = get_collection("courses", config_name, "secondary")
    courses_mongo3 = get_collection("courses", config_name, "mongo3")
    students_primary = get_collection("students", config_name, "primary")
    students_secondary = get_collection("students", config_name, "secondary")
    students_mongo3 = get_collection("students", config_name, "mongo3")

    course_id = f"C001-{config_name}"
    course_name = "Programming Methodology"
    student_id = f"S004-{config_name}"
    course_query = {"course_id": course_id}
    student_query = {"student_id": student_id}

    reset_document("courses", "course_id", make_course(course_id, course_name))
    reset_document("students", "student_id",
                   make_student(student_id, "Feyd Johnston", enrollment_note=None))

    stall_mongo3()
    try:
        # Without a session: read the course, then write a note based on that read
        status = read_field(courses_secondary, course_query, "course_status")
        show("Read course_status:", status)
        students_primary.update_one(student_query, {"$set": {
            "enrollment_note": f"Enrolled in {course_name} while course_status was: {status}"
        }})

        show("mongo3 course_status:", read_field(courses_mongo3, course_query, "course_status"))
        show("mongo3 enrollment_note:", read_field(students_mongo3, student_query, "enrollment_note"))
        show("Primary enrollment_note:", read_field(students_primary, student_query, "enrollment_note"))

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            status = read_field(courses_secondary, course_query, "course_status", session=session)
            show("Session read course_status:", status)
            students_primary.update_one(student_query, {"$set": {
                "enrollment_note": f"Session enrollment, course_status: {status}"
            }}, session=session)
            note = read_field(students_secondary, student_query, "enrollment_note", session=session)
            show("Session read-back:", note)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

## Check the Cluster

Resume mongo3 first, in case a previous run was interrupted while it was paused.

In [28]:
unstall_mongo3()
check_cluster()

Replica set:		 dsa5208
Primary:		 mongo1:27017
Writable primary:	 True


mongo1:27017 PRIMARY 2026-09-12 06:20:49
mongo2:27017 SECONDARY 2026-09-12 06:20:39
mongo3:27017 SECONDARY 2026-09-12 06:20:39




## Experiment 1: Read-your-writes

After the client writes a new `student_status`, repeated reads from secondaries are checked for the old value.

### Prediction

**Read-your-writes** requires that a client which has written a value never subsequently reads an
older one.

| Config | Session | Prediction |
|---|---|---|
| `w="majority"`, `readConcern="majority"` | `False` | **Violated** Majority write/read concern makes the write durable and the read safe to trust once it arrives, but neither says anything about which node answers the read. Routed to `ReadPreference.SECONDARY`, the read can still land on a node that hasn't replicated the write yet, and comes back with the old value. |
| `w="majority"`, `readConcern="majority"` | `True` | **Upheld** The session's timestamp forces the read to wait for the secondary to catch up if it hasn't already, so even on the rare occasion replication lags, the read still reflects the write rather than racing it. |
| `w="majority"`, `readConcern="local"` | `False` | **Violated** The write is still durable, but `local` read concern removes even the "safe to trust once it arrives" guarantee, so a lagging secondary just returns whatever it locally has, no different from the majority/majority case here since routing is still the actual problem. |
| `w="majority"`, `readConcern="local"` | `True` | **Upheld** The session's timestamp requirement doesn't depend on read concern level, it just needs the secondary to have replayed far enough, so this holds the same way as majority/majority with a session. |
| `w=1`, `readConcern="majority"` | `False` | **Violated** `w=1` acknowledges the write before it's replicated anywhere, so there's an even wider window where a secondary hasn't caught up yet. `readConcern="majority"` only guarantees the *value returned* won't later be rolled back, it doesn't guarantee that value is the *latest* one. |
| `w=1`, `readConcern="majority"` | `True` | **Upheld** Same mechanism as the other session cases, the read waits for the secondary to reach the write's timestamp regardless of write/read concern level. |
| `w=1`, `readConcern="local"` | `False` | **Violated** `w=1` acknowledges the write before it's replicated anywhere, and `local` reads return whatever's on the node with no freshness check, so a read that lands on a secondary microseconds behind can still return the old value. This is the config most likely to expose a violation even under normal conditions. |
| `w=1`, `readConcern="local"` | `True` | **Upheld** The session's requirement doesn't care about read/write concern level, it just makes the read wait until the secondary has replayed up to the write's timestamp, which under normal operation should only take milliseconds. |

**What to look for:** stale values in the no-session reads, and `TIMEOUT` instead of stale values in the session reads.

In [29]:
run_read_your_writes("majority")

=== Read-your-writes | w=majority, readConcern=majority ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1 Undergraduate
    2 Undergraduate
    3 Undergraduate
    4 Undergraduate
    5 Undergraduate
    6 Undergraduate
    7 Undergraduate
    8      Graduate
    9      Graduate
   10 Undergraduate
RESULTS: VIOLATION: 8 stale, 0 timed out, out of 10 reads


mongo1:27017 PRIMARY 2026-09-12 06:21:01
mongo2:27017 SECONDARY 2026-09-12 06:20:59
mongo3:27017 SECONDARY 2026-09-12 06:20:59


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1 Post Doc
    2 Post Doc
    3  TIMEOUT
    4 Post Doc
    5 Post Doc
    6 Post Doc
    7 Post Doc
    8 Post Doc
    9  TIMEOUT
   10 Post Doc
RESULTS: no violation in this run: 0 stale, 2 timed out, out of 10 reads


In [30]:
run_read_your_writes("majority-local")

=== Read-your-writes | w=majority, readConcern=local ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1 Undergraduate
    2      Graduate
    3 Undergraduate
    4      Graduate
    5      Graduate
    6 Undergraduate
    7 Undergraduate
    8 Undergraduate
    9 Undergraduate
   10      Graduate
RESULTS: VIOLATION: 6 stale, 0 timed out, out of 10 reads


mongo1:27017 PRIMARY 2026-09-12 06:21:26
mongo2:27017 SECONDARY 2026-09-12 06:21:24
mongo3:27017 SECONDARY 2026-09-12 06:21:24


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1  TIMEOUT
    2 Post Doc
    3 Post Doc
    4 Post Doc
    5  TIMEOUT
    6 Post Doc
    7 Post Doc
    8  TIMEOUT
    9 Post Doc
   10  TIMEOUT
RESULTS: no violation in this run: 0 stale, 4 timed out, out of 10 reads


In [31]:
run_read_your_writes("w1-majority")

=== Read-your-writes | w=1, readConcern=majority ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1 Undergraduate
    2 Undergraduate
    3 Undergraduate
    4 Undergraduate
    5 Undergraduate
    6 Undergraduate
    7 Undergraduate
    8 Undergraduate
    9 Undergraduate
   10 Undergraduate
RESULTS: VIOLATION: 10 stale, 0 timed out, out of 10 reads


mongo1:27017 PRIMARY 2026-09-12 06:21:54
mongo2:27017 SECONDARY 2026-09-12 06:21:52
mongo3:27017 SECONDARY 2026-09-12 06:21:52


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1  TIMEOUT
    2  TIMEOUT
    3 Post Doc
    4 Post Doc
    5  TIMEOUT
    6  TIMEOUT
    7  TIMEOUT
    8 Post Doc
    9  TIMEOUT
   10 Post Doc
RESULTS: no violation in this run: 0 stale, 6 timed out, out of 10 reads


In [32]:
run_read_your_writes("w1-local")

=== Read-your-writes | w=1, readConcern=local ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1      Graduate
    2      Graduate
    3      Graduate
    4      Graduate
    5      Graduate
    6 Undergraduate
    7 Undergraduate
    8 Undergraduate
    9      Graduate
   10 Undergraduate
RESULTS: VIOLATION: 4 stale, 0 timed out, out of 10 reads


mongo1:27017 PRIMARY 2026-09-12 06:22:14
mongo2:27017 SECONDARY 2026-09-12 06:22:12
mongo3:27017 SECONDARY 2026-09-12 06:22:12


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1  TIMEOUT
    2  TIMEOUT
    3 Post Doc
    4 Post Doc
    5 Post Doc
    6 Post Doc
    7 Post Doc
    8 Post Doc
    9  TIMEOUT
   10  TIMEOUT
RESULTS: no violation in this run: 0 stale, 4 timed out, out of 10 reads


### Interpretation

_Compare the two configs here._

## Experiment 2: Monotonic Reads

Seniority of student is advanced from Year 1 to Year 5 while mongo3 is paused, then read repeatedly from secondaries.

| Config | Session | Prediction |
|---|---|---|
| `w="majority"`, `readConcern="majority"` | `False` | **Violated** Once `mongo3` is stalled, it can no longer advance its own majority-commit point, so every read against it keeps returning `Year 1`, the last value it could confirm as majority-committed before the stall. Reads that land on `mongo2` show `Year 5`, so the sequence across nodes goes back and forth between the two, with `mongo3` never moving. |
| `w="majority"`, `readConcern="majority"` | `True` | **Upheld** Once a read has come back with `Year 5`, the session requires any later read to be at least that recent. `mongo3` can't satisfy that while stalled, so a read routed there blocks and returns TIMEOUT instead of going backwards. |
| `w="majority"`, `readConcern="local"` | `False` | **Violated** Without the majority-commit check, `mongo3` just returns whatever it currently has, so as the seniority field is updated step by step, `mongo3` can show any of the intermediate years it happened to apply before the stall took effect, not just `Year 1`. |
| `w="majority"`, `readConcern="local"` | `True` | **Upheld** The session's requirement doesn't depend on read concern level, so `mongo3` still can't satisfy it and the read times out instead of returning an older year. |
| `w=1`, `readConcern="majority"` | `False` | **Violated** `w=1` only changes how quickly the write is acknowledged, not how the secondary decides what's safe to return, so `mongo3` still freezes at its last majority-committed year and returns it every time. |
| `w=1`, `readConcern="majority"` | `True` | **Upheld** The session's timestamp requirement is unaffected by write concern, so this holds the same way as the other session cases. |
| `w=1`, `readConcern="local"` | `False` | **Violated** `local` reads return whatever's currently stored with no freshness check, and `w=1` lets writes race ahead of replication, so `mongo3` can flicker between several different years depending on exactly what it managed to apply before the stall. |
| `w=1`, `readConcern="local"` | `True` | **Upheld** The session's timestamp requirement still applies, so we get a timeout when trying to read stale data from `mongo3`, not a step backwards. |

**What to look for:** reads that go back from Year 5 to Year 1 without a session, and whether the session prevents this.

In [33]:
run_monotonic_reads("majority")

=== Monotonic reads | w=majority, readConcern=majority ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 1
    2 Year 5
    3 Year 1
    4 Year 1
    5 Year 5
    6 Year 1
    7 Year 5
    8 Year 5
    9 Year 1
   10 Year 1
RESULTS: VIOLATION: 5 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 1
    2  Year 5
    3  Year 5
    4 TIMEOUT
    5  Year 5
    6  Year 5
    7  Year 5
    8 TIMEOUT
    9  Year 5
   10  Year 5
RESULTS: no violation in this run: 0 went backwards, 2 timed out, out of 10 reads


In [34]:
run_monotonic_reads("majority-local")

=== Monotonic reads | w=majority, readConcern=local ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 5
    2 Year 1
    3 Year 1
    4 Year 1
    5 Year 1
    6 Year 5
    7 Year 1
    8 Year 5
    9 Year 5
   10 Year 5
RESULTS: VIOLATION: 5 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 5
    2  Year 5
    3 TIMEOUT
    4 TIMEOUT
    5  Year 5
    6 TIMEOUT
    7  Year 5
    8  Year 5
    9  Year 5
   10 TIMEOUT
RESULTS: no violation in this run: 0 went backwards, 4 timed out, out of 10 reads


In [35]:
run_monotonic_reads("w1-majority")

=== Monotonic reads | w=1, readConcern=majority ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 1
    2 Year 1
    3 Year 1
    4 Year 1
    5 Year 1
    6 Year 1
    7 Year 1
    8 Year 1
    9 Year 1
   10 Year 1
RESULTS: no violation in this run: 0 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read  value
    1 Year 1
    2 Year 1
    3 Year 1
    4 Year 1
    5 Year 1
    6 Year 1
    7 Year 1
    8 Year 1
    9 Year 1
   10 Year 1
RESULTS: no violation in this run: 0 went backwards, 0 timed out, out of 10 reads


In [36]:
run_monotonic_reads("w1-local")

=== Monotonic reads | w=1, readConcern=local ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 5
    2 Year 1
    3 Year 1
    4 Year 5
    5 Year 1
    6 Year 1
    7 Year 1
    8 Year 5
    9 Year 1
   10 Year 1
RESULTS: VIOLATION: 7 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 1
    2  Year 5
    3 TIMEOUT
    4 TIMEOUT
    5 TIMEOUT
    6 TIMEOUT
    7 TIMEOUT
    8  Year 5
    9 TIMEOUT
   10 TIMEOUT
RESULTS: no violation in this run: 0 went backwards, 7 timed out, out of 10 reads


### Interpretation

_Compare the two configs here._

## Experiment 3: Monotonic Writes

Two degrees are appended to `academic_history` while mongo3 is paused.

| Config | Prediction |
|---|---|
| `w="majority"`, `readConcern="majority"` | **Upheld** Both writes go to the same primary, so they land in one ordered oplog. `mongo3` is stalled, not diverging, so once it resumes it just replays the oplog from where it left off, passing through `[B.Sc.]` before `[B.Sc., M.Sc.]`, never the reverse. |
| `w="majority"`, `readConcern="local"` | **Upheld** Read concern only affects what's safe to return, not the order writes get applied in. `mongo3` still replays the oplog in order once it resumes. |
| `w=1`, `readConcern="majority"` | **Upheld** `w=1` only weakens how quickly the write is acknowledged, it doesn't change how `mongo3` applies the oplog once it catches up. |
| `w=1`, `readConcern="local"` | **Upheld** Ordering comes from oplog replay, not from write or read concern, so this holds the same way as the other three configs. |


**What to look for:** whether mongo3 applies the writes in the same order as the primary once it resumes.

In [37]:
run_monotonic_writes("majority")

=== Monotonic writes | w=majority, readConcern=majority ===
Primary:                       ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                                                                 value
    1 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    2                                              [B.Sc. Computer Science]
    3 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    4 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    5 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    6 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    7 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    8 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    9                                              [B.

In [38]:
run_monotonic_writes("majority-local")

=== Monotonic writes | w=majority, readConcern=local ===
Primary:                       ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                                                                 value
    1                                              [B.Sc. Computer Science]
    2                                              [B.Sc. Computer Science]
    3                                              [B.Sc. Computer Science]
    4 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    5 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    6                                              [B.Sc. Computer Science]
    7                                              [B.Sc. Computer Science]
    8                                              [B.Sc. Computer Science]
    9 [B.Sc. Computer Science, M.Sc. Data Science, PhD In

In [39]:
run_monotonic_writes("w1-majority")

=== Monotonic writes | w=1, readConcern=majority ===
Primary:                       ['B.Sc. Computer Science']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                    value
    1 [B.Sc. Computer Science]
    2 [B.Sc. Computer Science]
    3 [B.Sc. Computer Science]
    4 [B.Sc. Computer Science]
    5 [B.Sc. Computer Science]
    6 [B.Sc. Computer Science]
    7 [B.Sc. Computer Science]
    8 [B.Sc. Computer Science]
    9 [B.Sc. Computer Science]
   10 [B.Sc. Computer Science]
mongo3 (resumed):              ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']


In [40]:
run_monotonic_writes("w1-local")

=== Monotonic writes | w=1, readConcern=local ===
Primary:                       ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                                                                 value
    1 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    2 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    3 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    4                                              [B.Sc. Computer Science]
    5                                              [B.Sc. Computer Science]
    6 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    7 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    8                                              [B.Sc. Computer Science]
    9 [B.Sc. Computer Science, M.Sc. Data Science, PhD Informati

### Interpretation

_Compare the two configs here._

## Experiment 4: Writes-follow-reads

The client reads a course and writes an enrollment note based on what it read.

| Config | Session | Prediction |
|---|---|---|
| `w="majority"`, `readConcern="majority"` | `False` | **Violated** If the read is routed to the stalled `mongo3`, it returns the course's old status, one that's already been superseded on the primary. The note the client writes is then based on stale information, and it's committed after the real update in the oplog. The read-then-write chain is inverted relative to what actually happened. |
| `w="majority"`, `readConcern="majority"` | `True` | **Upheld** Same as the above session cases, where causal consistency is set to `True` and the rule is upheld |
| `w="majority"`, `readConcern="local"` | `False` | **Violated** `mongo3` just returns whatever it currently has, still the old course status, so the write is based on stale information regardless of read concern here. |
| `w="majority"`, `readConcern="local"` | `True` | **Upheld** Same as the above session cases, where causal consistency is set to `True` and the rule is upheld |
| `w=1`, `readConcern="majority"` | `False` | **Violated** `w=1` doesn't change what a stalled secondary can return, `mongo3` is still stuck on the old course status either way. |
| `w=1`, `readConcern="majority"` | `True` | **Upheld** Same as the above session cases, where causal consistency is set to `True` and the rule is upheld |
| `w=1`, `readConcern="local"` | `False` | **Violated** No freshness check at all on the read, so the stale status comes back immediately and the write is built on it straight away. |
| `w=1`, `readConcern="local"` | `True` | **Upheld** Same as the above session cases, where causal consistency is set to `True` and the rule is upheld |


**What to look for:** whether a node can show the enrollment note without the course state it depended on.

In [41]:
run_writes_follow_reads("majority")

=== Writes-follow-reads | w=majority, readConcern=majority ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        None
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             Session enrollment, course_status: ACTIVE


In [42]:
run_writes_follow_reads("majority-local")

=== Writes-follow-reads | w=majority, readConcern=local ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        None
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             TIMEOUT


In [43]:
run_writes_follow_reads("w1-majority")

=== Writes-follow-reads | w=1, readConcern=majority ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        None
Primary enrollment_note:       None
Session read course_status:    ACTIVE
Session read-back:             TIMEOUT


In [44]:
run_writes_follow_reads("w1-local")

=== Writes-follow-reads | w=1, readConcern=local ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        None
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             Session enrollment, course_status: ACTIVE


### Interpretation

_Compare the two configs here._